In [9]:
import threading
import time
import random
import string
import sys
import asyncio
import ipywidgets as widgets
from IPython.display import display
from playwright.sync_api import sync_playwright

# Shared state between the UI and the Playwright background thread
bot_state = {
    "wpm": 45,
    "running": True
}

def calculate_delay(wpm):
    safe_wpm = max(1, wpm)
    return 60.0 / (safe_wpm * 5)

def simulate_human_typing(page, input_locator, text, error_chance=0.03):
    current_wpm = max(15, bot_state["wpm"] * 0.5) 
    
    for char in text:
        if not bot_state["running"]:
            print("\nTyping sequence halted.")
            return

        target_wpm = bot_state["wpm"] 
        ramp_up_step = (target_wpm - current_wpm) / max(1, (len(text) * 0.3))
        
        if current_wpm < target_wpm:
            current_wpm += ramp_up_step
            
        base_delay = calculate_delay(current_wpm)
        
        if random.random() < error_chance and char.isalpha():
            wrong_char = random.choice(string.ascii_lowercase)
            input_locator.type(wrong_char)
            time.sleep(random.uniform(0.15, 0.3)) 
            page.keyboard.press("Backspace")
            time.sleep(random.uniform(0.05, 0.1)) 
            
        jitter = random.uniform(-0.02, 0.03)
        actual_delay = max(0.01, base_delay + jitter)
        
        input_locator.type(char)
        time.sleep(actual_delay)

def run_playwright_engine():
    # ---------------------------------------------------------
    # CRITICAL FIX: Force a Proactor loop strictly within this thread
    # ---------------------------------------------------------
    if sys.platform == 'win32':
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        asyncio.set_event_loop(asyncio.new_event_loop())
    
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=False, channel="chrome")
            context = browser.new_context()
            page = context.new_page()
            
            page.goto("https://play.typeracer.com/")
            
            while bot_state["running"]:
                try:
                    if page.locator("text='Enter a typing race'").is_visible():
                        page.locator("text='Enter a typing race'").click()
                    elif page.locator("a.raceAgainLink").is_visible():
                        page.locator("a.raceAgainLink").click()
                    
                    input_box = page.locator("input.txtInput")
                    
                    box_enabled = False
                    for _ in range(60): 
                        if not bot_state["running"]:
                            break
                        if input_box.count() > 0 and not input_box.is_disabled():
                            box_enabled = True
                            break
                        time.sleep(1)
                    
                    if not box_enabled or not bot_state["running"]:
                        continue

                    input_box.focus()
                    
                    spans = page.locator("//div[contains(@style, 'monospace')]//span[@unselectable='on']").all_inner_texts()
                    full_text = "".join(spans).replace('\xa0', ' ') 
                    
                    simulate_human_typing(page, input_box, full_text)
                    
                    if bot_state["running"]:
                        time.sleep(random.uniform(4.0, 7.0)) 
                    
                except Exception as inner_e:
                    if "Target page, context or browser has been closed" in str(inner_e):
                        print("\nBrowser closed manually. Engine shutting down.")
                        bot_state["running"] = False
                        break
                    time.sleep(2)
    except Exception as e:
        print(f"Engine failure: {e}")

# --- UI Controller Setup ---
def start_bot_with_ui():
    bot_state["running"] = True
    bot_state["wpm"] = 45
    
    speed_slider = widgets.IntSlider(
        value=45, min=10, max=150, step=1,
        description='Target WPM:',
        continuous_update=True
    )
    
    def on_speed_change(change):
        bot_state["wpm"] = change['new']
    speed_slider.observe(on_speed_change, names='value')
    
    stop_button = widgets.Button(
        description='Stop Engine',
        button_style='danger',
        tooltip='Click to stop the automation'
    )
    
    def on_stop_clicked(b):
        bot_state["running"] = False
        stop_button.description = 'Stopping...'
        stop_button.disabled = True
    stop_button.on_click(on_stop_clicked)
    
    display(widgets.HBox([speed_slider, stop_button]))
    
    engine_thread = threading.Thread(target=run_playwright_engine, daemon=True)
    engine_thread.start()

start_bot_with_ui()